# 55 · B-trees and block storage

*Trees* · 9 exercises

Count block reads instead of steps, and pack many keys per node to keep the tree shallow.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ch-55) · [Answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ch-55) · [Concept guide](https://bloomcode.github.io/dsa/guide/#ch-55)

**How to work:** run the setup cell once. For each exercise, replace the stub in its code cell, run the cell, then run the test cell under it. Later exercises often reuse earlier ones, so go in order. Stuck? Each exercise links to its tested solution in the answer key.

In [ ]:
# Setup: run once.
import bisect
import math
import random

try:
    import pytest
except ImportError:  # the tests only need pytest.raises; a small stand-in keeps the notebook self-contained
    class _Raises:
        def __init__(self, kind):
            self.kind = kind

        def __enter__(self):
            return self

        def __exit__(self, kind, value, tb):
            if kind is None:
                raise AssertionError(f"did not raise {self.kind.__name__}")
            return issubclass(kind, self.kind)

    class pytest:  # noqa: N801
        raises = _Raises

### 55.1 · Count I/Os, not steps  `Core`

**Task.** Memory moves in blocks of B items. Write `scan_ios(n, B)` for reading n items in order, and `random_reads_ios(k)` for k reads that each land in a different block.

**Example.** `scan_ios(1000, 64)` -> `16`; `random_reads_ios(1000)` -> `1000`

**Big O.** A scan costs O(N / B) I/Os; scattered access costs one I/O per item. On disk, I/Os dominate the running time.

*Its test runs further down, once the code it relies on has been defined.*

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-55-1) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-55-1)

In [ ]:
# 55.1 Count I/Os, not steps
def scan_ios(n, B):
    raise NotImplementedError


def random_reads_ios(k):
    raise NotImplementedError

### 55.2 · Binary search over blocks  `Core`

**Task.** Write `ExternalArray(data, B)` whose `read_block(i)` returns block i and counts one I/O, and `external_binary_search(ext, target)` that binary-searches the blocks by their first item, then searches inside one block in memory.

**Example.** 10,000 sorted items with B = 100 -> a lookup costs about 8 I/Os, versus about 14 for plain binary search touching one block per probe

**Big O.** O(log(N / B)) I/Os, a small saving; the B-tree (55.4) does much better with O(log_B N).

*Its test runs further down, once the code it relies on has been defined.*

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-55-2) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-55-2)

In [ ]:
# 55.2 Binary search over blocks
class ExternalArray:
    def __init__(self, data, B):
        raise NotImplementedError

    def read_block(self, i):
        raise NotImplementedError


def external_binary_search(ext, target):
    raise NotImplementedError

### 55.3 · Lookups: BST versus B-tree  `Core`

**Task.** Write `bst_ios(n)` (one I/O per level of a balanced binary tree, log2 N) and `btree_ios(n, B)` (log base B of N) and compare them for a billion keys.

**Example.** `bst_ios(10**9)` -> `30`; `btree_ios(10**9, 1000)` -> `3`

**Big O.** O(log_2 N) versus O(log_B N) I/Os: a factor of log2 B fewer.

*Its test runs further down, once the code it relies on has been defined.*

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-55-3) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-55-3)

In [ ]:
# 55.3 Lookups: BST versus B-tree
def bst_ios(n):
    raise NotImplementedError


def btree_ios(n, B):
    raise NotImplementedError

### 55.4 · B-tree nodes and search  `Core`

**Task.** Create `BTreeNode(leaf)` with sorted `keys` and `children`, and `BTree(t)` with a `search(key)` that binary-searches each node's keys and descends into the right child.

**Example.** in a B-tree holding 1..100 -> `search(42)` is `True`, `search(101)` is `False`

**Big O.** O(log_t N) nodes visited, O(log t) comparisons in each.

*Its test runs further down, once the code it relies on has been defined.*

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-55-4) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-55-4)

In [ ]:
# 55.4 B-tree nodes and search
class BTreeNode:
    def __init__(self, leaf=True):
        raise NotImplementedError


class BTree:
    def __init__(self, t=3):
        raise NotImplementedError

    def search(self, key, node=None):
        raise NotImplementedError

### 55.5 · Insert with proactive splits  `Core`

**Task.** Add `insert(key)`: if the root is full (2t - 1 keys), split it and grow a new root. Walk down, splitting any full child before entering it, so a leaf always has room.

**Example.** t = 2, insert 1..10 -> `validate()` passes and every leaf sits at the same depth

**Big O.** O(t log_t N) CPU, O(log_t N) node reads. The tree grows only at the root, which keeps it perfectly balanced.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-55-5) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-55-5)

In [ ]:
# 55.5 Insert with proactive splits
def split_child(self, parent, i):
    raise NotImplementedError

def insert(self, key):
    raise NotImplementedError


BTree.split_child = split_child
BTree.insert = insert

In [ ]:
# Test helpers: run once.
def btree(t, keys):
    tree = BTree(t)
    for k in keys:
        tree.insert(k)
    return tree

In [ ]:
def test_ex_55_1():
    assert scan_ios(1000, 64) == 16 and random_reads_ios(1000) == 1000


test_ex_55_1()
print('55.1 passed')

In [ ]:
def test_ex_55_2():
    ext = ExternalArray(range(0, 20000, 2), 100)
    assert external_binary_search(ext, 4242) == 2121 and ext.ios <= 9
    assert external_binary_search(ExternalArray(range(0, 20000, 2), 100), 4243) == -1
    assert external_binary_search(ExternalArray(range(10), 3), 0) == 0
    assert external_binary_search(ExternalArray(range(10), 3), 9) == 9


test_ex_55_2()
print('55.2 passed')

In [ ]:
def test_ex_55_3():
    assert bst_ios(10**9) == 30 and btree_ios(10**9, 1000) == 3


test_ex_55_3()
print('55.3 passed')

In [ ]:
def test_ex_55_4():
    t = BTree(2)
    assert not t.search(1)
    left, right = BTreeNode(leaf=True), BTreeNode(leaf=True)
    left.keys, right.keys = [1, 2], [4, 5]
    t.root = BTreeNode(leaf=False)
    t.root.keys, t.root.children = [3], [left, right]
    assert all(t.search(k) for k in (1, 2, 3, 4, 5)) and not t.search(0) and not t.search(6)


test_ex_55_4()
print('55.4 passed')

In [ ]:
def test_ex_55_5():
    t = btree(2, range(1, 11))
    assert t.search(7) and not t.search(11) and all(t.search(k) for k in range(1, 11))
    assert all(len(n.keys) <= 3 for n in [t.root] + t.root.children)


test_ex_55_5()
print('55.5 passed')

### 55.6 · Validate and walk in order  `Core`

**Task.** Add `inorder()` returning all keys sorted, and `validate()` that checks key counts per node, sorted keys, child counts, and that all leaves share one depth. Return the height.

**Example.** after random inserts and deletes -> `inorder()` equals the sorted set of keys and `validate()` returns the height

**Big O.** O(N).

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-55-6) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-55-6)

In [ ]:
# 55.6 Validate and walk in order
def inorder(self, node=None):
    raise NotImplementedError

def validate(self):
    raise NotImplementedError


BTree.inorder = inorder
BTree.validate = validate

In [ ]:
def test_ex_55_6():
    t = BTree(2)
    for k in range(1, 11):
        t.insert(k)
        t.validate()
    assert t.inorder() == list(range(1, 11))


test_ex_55_6()
print('55.6 passed')

### 55.8 · Range queries, like a database index  `Practice`

**Task.** Add `range_query(lo, hi)` returning the keys in [lo, hi] in order, skipping whole subtrees that cannot contain them.

**Example.** B-tree of 1..100 -> `range_query(40, 45)` is `[40, 41, 42, 43, 44, 45]`

**Big O.** O(log_t N + K / t) node visits for K results. This is why databases index with B-trees.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-55-8) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-55-8)

In [ ]:
# 55.8 Range queries, like a database index
def range_query(self, lo, hi, node=None, out=None):
    raise NotImplementedError


BTree.range_query = range_query

In [ ]:
def test_ex_55_8():
    t = btree(3, range(1, 101))
    assert t.range_query(40, 45) == [40, 41, 42, 43, 44, 45]
    assert t.range_query(0, 3) == [1, 2, 3] and t.range_query(200, 300) == []


test_ex_55_8()
print('55.8 passed')

### 55.9 · Deletion  `Stretch`

**Task.** Add `delete(key)`. Before descending into a child with only t - 1 keys, top it up by borrowing from a sibling or merging with one. Delete from a leaf directly; for an internal key, replace it with its predecessor or successor, or merge. Shrink the root when it empties.

**Example.** t = 3, insert 1..200, delete the odd keys in shuffled order -> `validate()` passes and `inorder()` is the even keys

**Big O.** O(log_t N) node visits.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-55-9) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-55-9)

In [ ]:
# 55.9 Deletion
def delete(self, key):
    raise NotImplementedError

def _delete(self, node, key):
    raise NotImplementedError

def _merge(self, node, i):
    raise NotImplementedError


BTree.delete = delete
BTree._delete = _delete
BTree._merge = _merge

In [ ]:
def test_ex_55_9():
    rng = random.Random(8)
    for t_deg in (2, 3, 4):
        t, keys = BTree(t_deg), set()
        for _ in range(1500):
            k = rng.randrange(400)
            if rng.random() < 0.55:
                t.insert(k)
                keys.add(k)
            else:
                assert t.delete(k) == (k in keys)
                keys.discard(k)
        t.validate()
        assert t.inorder() == sorted(keys)
    t = btree(3, range(1, 201))
    odds = list(range(1, 201, 2))
    random.Random(3).shuffle(odds)
    for k in odds:
        t.delete(k)
    t.validate()
    assert t.inorder() == list(range(2, 201, 2))


test_ex_55_9()
print('55.9 passed')

### 55.7 · How tall can a B-tree get?  `Core`

**Task.** Write `btree_height_bound(n, t)` returning the most levels a B-tree of minimum degree t with n keys can have: 1 + floor(log_t((n + 1) / 2)). Check it against real trees.

**Example.** `btree_height_bound(10**6, 50)` -> `4`; a real tree of 1..10,000 with t = 3 stays within its bound of 8

**Big O.** O(1). With t in the hundreds, a billion keys fit in 4 or 5 levels.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-55-7) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-55-7)

In [ ]:
# 55.7 How tall can a B-tree get?
def btree_height_bound(n, t):
    raise NotImplementedError

In [ ]:
def test_ex_55_7():
    assert btree_height_bound(10**6, 50) == 4
    assert btree(3, range(1, 10001)).validate() <= btree_height_bound(10000, 3)


test_ex_55_7()
print('55.7 passed')